# CharXiv Feature Selection, Checkpoint 2

This notebook turns the exploratory analysis into defensible keep-or-drop decisions, logged to
`results/feature_selection.csv`. The task is to predict failure for each of the three target models
with a separate classifier, where the baseline is the item metadata and the engineered features are the
twelve DeLeAn demand dimensions (the only engineered features). The selection funnel moves from domain
knowledge to leakage, then to low-information features, then to redundancy (correlation and the variance
inflation factor), then to model-based importance, and finally to a metadata ablation that asks whether
the demand dimensions beat a cheap metadata-only baseline. It uses the training items only,
paper-grouped cross-validation, and the failure-as-positive convention.

## 1. Setup

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src.features.preprocessing as P
from src.features.transformers import DEMAND_DIMS
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
RESULTS = ROOT / "results"

TARGETS = P.TARGET_MODELS
REAL = ["GPT-4o", "Claude-3-5-Sonnet"]        # the two non-control targets
con = P.connect(); train_ids, test_ids = P.get_split(con)
df = P.make_design_matrix(con, train_ids)      # one row per item
con.close()
item_dims = df.set_index("item_id")[DEMAND_DIMS]
decisions = []   # feature, type, decision, reason, key_stat
print("items:", len(df), "| targets:", TARGETS)

items: 800 | targets: ['GPT-4o', 'Claude-3-5-Sonnet', 'GPT-4o-Random']


## 2. Domain knowledge and leakage check

The candidate features are the twelve DeLeAn demand dimensions (engineered) and the item metadata (the
baseline candidate). On leakage, the features are item-intrinsic, capturing the chart's cognitive demand
and its metadata. No model's correctness is an input, because the ground-truth `answer`, the other
vision-language models' correctness, and the `Human` correctness are all excluded, so nothing about the
verdict leaks into the features.

In [2]:
print("Excluded from features (leakage / not item-intrinsic): ground-truth answer, other VLM")
print("correctness, and Human correctness. Kept candidates: 12 demand dims + item metadata.")

Excluded from features (leakage / not item-intrinsic): ground-truth answer, other VLM
correctness, and Human correctness. Kept candidates: 12 demand dims + item metadata.


## 3. Low-information features

The dimension `CL` is near-constant, so it is dropped. The dimensions `GS` and `KNf` are sparse but
kept, because sparse does not mean uninformative, as the importance results below confirm.

In [3]:
lowinfo = pd.DataFrame({"mean": item_dims.mean(), "std": item_dims.std(),
                        "pct_zero": (item_dims == 0).mean()}).loc[DEMAND_DIMS].round(3)
display(lowinfo)
for d in DEMAND_DIMS:
    if d == "CL":
        decisions.append({"feature": "CL", "type": "demand_dim", "decision": "drop",
                          "reason": "near-constant (~82% zeros, mean~0.19)",
                          "key_stat": f"pct_zero={lowinfo.loc['CL','pct_zero']}"})
    else:
        decisions.append({"feature": d, "type": "demand_dim", "decision": "keep",
                          "reason": "engineered demand dim", "key_stat": f"mean={lowinfo.loc[d,'mean']}"})

,mean,std,pct_zero
VL,2.281,0.642,0.008
AS,2.645,0.540,0.000
MCr,1.644,0.684,0.016
MCu,0.670,0.644,0.416
MA,2.399,1.376,0.126
VO,1.170,0.376,0.000
AT,2.565,0.521,0.000
GS,0.810,1.173,0.611
QLl,1.004,0.620,0.164
KNf,1.168,0.906,0.221


## 4. Redundancy, correlation and the variance inflation factor

The variance inflation factor is computed with a small scikit-learn helper that regresses each dimension
on the others and takes 1/(1 − R²), so there is no `statsmodels` dependency.

In [4]:
def vif(frame):
    out = {}
    for col in frame.columns:
        r2 = LinearRegression().fit(frame.drop(columns=col), frame[col]).score(frame.drop(columns=col), frame[col])
        out[col] = round(1.0 / max(1e-9, 1 - r2), 2)
    return pd.Series(out)
kept_dims = [d for d in DEMAND_DIMS if d != "CL"]
v = vif(item_dims[kept_dims]).sort_values(ascending=False)
print("VIF (all < 5 -> low multicollinearity; keep all kept dims):")
display(v.to_frame("VIF"))

VIF (all < 5 -> low multicollinearity; keep all kept dims):


,VIF
MCr,1.76
QLl,1.64
AS,1.60
QLq,1.59
VL,1.44
MA,1.44
MCu,1.42
VO,1.41
AT,1.37
KNf,1.21


## 5. Model-based diagnostics (supporting evidence)

This section reports RandomForest importance for the demand dimensions, predicting failure for each real
target. It is supporting evidence rather than decisive. The random control is omitted here because its
outcome does not track reasoning demand.

In [5]:
imp = {}
for t in REAL:
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=1)
    rf.fit(df[kept_dims], df[P.ycol(t)])
    imp[t] = pd.Series(rf.feature_importances_, index=kept_dims)
imp = pd.DataFrame(imp).round(3).sort_values(REAL[0], ascending=False)
display(imp)
print("GS/MA/MCu are sparse but among the more important dims -> keep (sparse != uninformative).")

,GPT-4o,Claude-3-5-Sonnet
MCu,0.171,0.120
MA,0.137,0.153
MCr,0.109,0.099
GS,0.107,0.118
KNf,0.095,0.104
AS,0.082,0.085
VL,0.069,0.091
QLq,0.068,0.073
AT,0.059,0.059
QLl,0.059,0.062


GS/MA/MCu are sparse but among the more important dims -> keep (sparse != uninformative).


## 6. Metadata ablation, do the demand dimensions beat a metadata-only baseline?

This ablation reports grouped cross-validation ROC-AUC (logistic regression) for the two real targets
under three feature sets: metadata only (the baseline), demand dimensions only, and demand dimensions
plus metadata. If the demand dimensions clearly beat metadata and adding metadata back does not help,
metadata is dropped from the final feature set.

In [6]:
def cv_auc(make_pre, target, n_splits=5, n_repeats=3):
    X, y, g = df, df[P.ycol(target)].to_numpy(), df.groups.to_numpy()
    aucs = []
    for r in range(n_repeats):
        for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=r).split(X, y, g):
            pipe = Pipeline([("pre", make_pre()), ("clf", LogisticRegression(max_iter=1000))])
            pipe.fit(X.iloc[tr], y[tr])
            aucs.append(roc_auc_score(y[te], pipe.predict_proba(X.iloc[te])[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))

meta_only = lambda: P.build_preprocessor(include_demand_dims=False, include_metadata=True)
dims_only = lambda: P.build_preprocessor(include_demand_dims=True,  include_metadata=False)
dims_meta = lambda: P.build_preprocessor(include_demand_dims=True,  include_metadata=True)
rows = []
for t in REAL:
    m, d, b = cv_auc(meta_only, t), cv_auc(dims_only, t), cv_auc(dims_meta, t)
    rows.append({"target": t, "metadata_only": round(m[0], 4), "demand_only": round(d[0], 4),
                 "demand+metadata": round(b[0], 4), "meta_lift_over_demand": round(b[0]-d[0], 4),
                 "fold_sd": round(d[1], 4)})
ablation = pd.DataFrame(rows)
display(ablation.set_index("target"))
mean_lift = float(ablation["meta_lift_over_demand"].mean()); mean_sd = float(ablation["fold_sd"].mean())
meta_decision = "drop" if mean_lift < mean_sd else "keep"
for col in P.CATEGORICAL:
    decisions.append({"feature": col, "type": "item_metadata_baseline", "decision": meta_decision,
                      "reason": f"demand dims beat metadata-only; adding metadata to the dims lifts "
                                f"ROC-AUC by {mean_lift:+.4f} (< 1 fold SD)",
                      "key_stat": f"meta_lift_over_demand={mean_lift:+.4f}"})
print("The demand dims clearly beat metadata-only for both real models; metadata decision:", meta_decision)

,metadata_only,demand_only,demand+metadata,meta_lift_over_demand,fold_sd
target,,,,,
GPT-4o,0.5279,0.6758,0.6678,-0.0080,0.0336
Claude-3-5-Sonnet,0.5359,0.6275,0.6170,-0.0105,0.0490


The demand dims clearly beat metadata-only for both real models; metadata decision: drop


## 7. Decisions written to `results/feature_selection.csv`

In [7]:
fs = pd.DataFrame(decisions, columns=["feature", "type", "decision", "reason", "key_stat"])
fs.to_csv(RESULTS / "feature_selection.csv", index=False)
print("saved -> results/feature_selection.csv"); display(fs)
print("\nSummary: keep the 11 demand dims (drop CL); metadata ->", meta_decision,
      "-> the final feature set is the 11 demand dims, with metadata retained only as the baseline.")

saved -> results/feature_selection.csv


,feature,type,decision,reason,key_stat
0,VL,demand_dim,keep,engineered demand dim,mean=2.281
1,AS,demand_dim,keep,engineered demand dim,mean=2.645
2,MCr,demand_dim,keep,engineered demand dim,mean=1.644
3,MCu,demand_dim,keep,engineered demand dim,mean=0.67
4,MA,demand_dim,keep,engineered demand dim,mean=2.399
5,VO,demand_dim,keep,engineered demand dim,mean=1.17
6,AT,demand_dim,keep,engineered demand dim,mean=2.565
7,GS,demand_dim,keep,engineered demand dim,mean=0.81
8,QLl,demand_dim,keep,engineered demand dim,mean=1.004
9,KNf,demand_dim,keep,engineered demand dim,mean=1.168



Summary: keep the 11 demand dims (drop CL); metadata -> drop -> the final feature set is the 11 demand dims, with metadata retained only as the baseline.
